# 1. Carregar e limpar dados

## 1.1 Importações de bibliotecas e base de dados

In [292]:
import pandas as pd
import numpy as np

In [293]:
# Importando base de dados do campeonato brasileiro
df = pd.read_csv("data/campeonato-brasileiro-full.csv")

df.sample(5)

,ID,rodata,data,hora,mandante,visitante,formacao_mandante,formacao_visitante,tecnico_mandante,tecnico_visitante,vencedor,arena,mandante_Placar,visitante_Placar,mandante_Estado,visitante_Estado,arrecadacao
2693,2694,37,30/11/2008,17:00,Flamengo,Goias,NaN,NaN,NaN,NaN,-,Maracanã,3,3,RJ,GO,NaN
2070,2082,13,22/07/2007,16:00,Gremio,Flamengo,NaN,NaN,NaN,NaN,Gremio,Olímpico,1,0,RS,RJ,NaN
6883,6877,38,08/12/2019,16:02,Santos,Flamengo,4-3-3,4-2-3-1,J. Sampaoli Moya,J. Pinheiro de Jesus,Santos,Estádio Urbano Caldeira,4,0,SP,RJ,NaN
5845,5846,11,01/07/2017,16:02,Palmeiras,Gremio,4-3-3,4-2-3-1,A. Stival,Renato Gaúcho,Palmeiras,Estádio Municipal Paulo Machado de Carvalho,1,0,SP,RS,NaN
1237,1241,13,20/07/2005,20:30,Internacional,Goias,NaN,NaN,NaN,NaN,Goias,Beira Rio,2,3,RS,GO,NaN


## 1.2 Análise exploratória dos dados

In [294]:
# Verificando colunas do DataFrame
df.columns

Index(['ID', 'rodata', 'data', 'hora', 'mandante', 'visitante',
       'formacao_mandante', 'formacao_visitante', 'tecnico_mandante',
       'tecnico_visitante', 'vencedor', 'arena', 'mandante_Placar',
       'visitante_Placar', 'mandante_Estado', 'visitante_Estado',
       'arrecadacao'],
      dtype='str')

In [295]:
# Mantendo apenas as colunas que serão utilizadas
df = df[['rodata','data','mandante','visitante', 'mandante_Placar', 'visitante_Placar']]

In [296]:
# Contagem de linhas e colunas do DataFrame
df.shape

(9165, 6)

In [297]:
# Verificando tipos das colunas e valores nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9165 entries, 0 to 9164
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   rodata            9165 non-null   int64
 1   data              9165 non-null   str  
 2   mandante          9165 non-null   str  
 3   visitante         9165 non-null   str  
 4   mandante_Placar   9165 non-null   int64
 5   visitante_Placar  9165 non-null   int64
dtypes: int64(3), str(3)
memory usage: 429.7 KB


# 1.3 Limpeza e formatação

In [298]:
# Corrigindo coluna data para o tipo datetime
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')

In [299]:
# Renomeando colunas para facilitar a manipulação dos dados
df = df.rename(columns={'rodata':'rodada', 'data':'data', 'mandante':'mandante', 'visitante':'visitante', 'vencedor':'vencedor', 'mandante_Placar':'placar_mandante', 'visitante_Placar':'placar_visitante'})

In [ ]:
# Filtrando apenas os jogos a partir de 2006, ano em que o campeonato brasileiro passou a ser disputado no formato atual
df = df[df['data'] >= '2006-01-01']

In [301]:
# Nova quantidade de linhas do DataFrame após o filtro
df.shape

(7599, 6)

In [302]:
df.head()

,rodada,data,mandante,visitante,placar_mandante,placar_visitante
1566,1,2006-04-15,Juventude,Parana,1,0
1567,1,2006-04-15,Vasco,Internacional,1,1
1568,1,2006-04-16,Sao Paulo,Flamengo,1,0
1569,1,2006-04-16,Gremio,Corinthians,2,0
1570,1,2006-04-16,Botafogo-RJ,Fortaleza,1,0


In [ ]:
# Arrumando o índice do DataFrame após o filtro
df = df.reset_index(drop=True).sort_values(by=['data', 'rodada'], ascending=[True, True])

In [304]:
# Verificando o período de tempo dos dados
print(df['data'].min())
print(df['data'].max())

2006-04-15 00:00:00
2025-12-07 00:00:00


## 1.4 Criação de novas colunas

In [305]:
# Criando a coluna 'ano'
df['ano'] = df['data'].dt.year

In [333]:
# Contando quantos jogos tiveram no ano
df.groupby('ano')['rodada'].count()

ano
2006    380
2007    380
2008    380
2009    380
2010    380
2011    380
2012    380
2013    380
2014    380
2015    380
2016    379
2017    380
2018    380
2019    380
2020    268
2021    492
2022    380
2023    380
2024    380
2025    380
Name: rodada, dtype: int64

In [306]:
# Condições pontuação do time mandante
condicoes_pontuacao_mandante = [
    df['placar_mandante'] > df['placar_visitante'],
    df['placar_mandante'] == df['placar_visitante'],
    df['placar_visitante'] > df['placar_mandante']
]

# Quantidade de pontos baseado na condição
pontuacao = [3, 1, 0]

# Adição da coluna com as pontuações calculadas
df['pontos_mandante'] = np.select(condicoes_pontuacao_mandante, pontuacao)


In [307]:
# Condições pontuação do time visitante
condicoes_pontuacao_visitante = [
    df['placar_visitante'] > df['placar_mandante'],
    df['placar_mandante'] == df['placar_visitante'],
    df['placar_mandante'] > df['placar_visitante']
]

# Quantidade de pontos baseado na condição
pontuacao = [3, 1, 0]

# Adição da coluna com as pontuações calculadas
df['pontos_visitante'] = np.select(condicoes_pontuacao_visitante, pontuacao)


In [308]:
condicoes_resultado = [
    df['placar_mandante'] > df['placar_visitante'],
    df['placar_mandante'] == df['placar_visitante'],
    df['placar_mandante'] < df['placar_visitante']
]

resultados = ['M', 'E', 'V']

df['resultado'] = np.select(condicoes_resultado, resultados, default='Indefinido')

In [322]:
df.sample(5)

,rodada,data,mandante,visitante,placar_mandante,placar_visitante,ano,pontos_mandante,pontos_visitante,resultado
5741,5,2021-06-20,Fortaleza,Fluminense,1,1,2021,1,1,E
1993,10,2011-07-17,Ceara,America-MG,4,0,2011,3,0,M
601,23,2007-09-02,Corinthians,Santos,2,0,2007,3,0,M
6852,2,2024-04-17,Atletico-MG,Criciuma,1,1,2024,1,1,E
4092,30,2016-10-12,Santa Cruz,Corinthians,2,4,2016,0,3,V


# 2. Classificação de uma temporada específica (teste de sanidade)

## 2.1 Criando dataframes do time mandante

In [310]:
# Criando dataframe apenas com os jogos dos times mandante
df_mandante = df[['rodada', 'data', 'ano', 'mandante', 'placar_mandante', 'placar_visitante', 'pontos_mandante', 'visitante']]

# Renomeando as colunas
df_mandante = df_mandante.rename(columns={'rodada':'rodada', 'data':'data', 'ano':'ano', 'mandante':'time', 'placar_mandante':'gols_pro', 'placar_visitante':'gols_contra', 'pontos_mandante':'pontos', 'visitante':'adversario'})

In [311]:
condicoes_resultado_mandante = [
    df_mandante['gols_pro'] > df_mandante['gols_contra'],
    df_mandante['gols_pro'] == df_mandante['gols_contra'],
    df_mandante['gols_pro'] < df_mandante['gols_contra']
]

resultado_mandante = ['M', 'E', 'V']

In [312]:
df_mandante['mando'] = 1

df_mandante['resultado'] = np.select(condicoes_resultado_mandante, resultado_mandante, default='Indefinido')

df_mandante.head()

,rodada,data,ano,time,gols_pro,gols_contra,pontos,adversario,mando,resultado
0,1,2006-04-15,2006,Juventude,1,0,3,Parana,1,M
1,1,2006-04-15,2006,Vasco,1,1,1,Internacional,1,E
2,1,2006-04-16,2006,Sao Paulo,1,0,3,Flamengo,1,M
3,1,2006-04-16,2006,Gremio,2,0,3,Corinthians,1,M
4,1,2006-04-16,2006,Botafogo-RJ,1,0,3,Fortaleza,1,M


## 2.2 Criando dataframe com jogos do time visitante

In [313]:
# Criando dataframe apenas com os jogos dos times visitantes
df_visitante = df[['rodada', 'data', 'ano', 'visitante', 'placar_visitante', 'placar_mandante', 'pontos_visitante', 'mandante']]

# Renomeando as colunas
df_visitante = df_visitante.rename(columns={'rodada':'rodada', 'data':'data', 'ano':'ano', 'visitante':'time', 'placar_visitante':'gols_pro', 'placar_mandante':'gols_contra', 'pontos_visitante':'pontos', 'mandante':'adversario'})

In [314]:
condicoes_resultado_visitante = [
    df_visitante['gols_pro'] > df_visitante['gols_contra'],
    df_visitante['gols_pro'] == df_visitante['gols_contra'],
    df_visitante['gols_pro'] < df_visitante['gols_contra']
]

resultado_visitante = ['V', 'E', 'M']

In [315]:
df_visitante['mando'] = 0

df_visitante['resultado'] = np.select(condicoes_resultado_visitante, resultado_visitante, default='Indefinido')

df_visitante.head()

,rodada,data,ano,time,gols_pro,gols_contra,pontos,adversario,mando,resultado
0,1,2006-04-15,2006,Parana,0,1,0,Juventude,0,M
1,1,2006-04-15,2006,Internacional,1,1,1,Vasco,0,E
2,1,2006-04-16,2006,Flamengo,0,1,0,Sao Paulo,0,M
3,1,2006-04-16,2006,Corinthians,0,2,0,Gremio,0,M
4,1,2006-04-16,2006,Fortaleza,0,1,0,Botafogo-RJ,0,M


## 2.3 Juntando dataframes de mandante e visitante em um

In [316]:
# Juntando os dois dataframes em um único
df_tabela_total = pd.concat([df_mandante, df_visitante])
df_tabela_total.sample(5)

,rodada,data,ano,time,gols_pro,gols_contra,pontos,adversario,mando,resultado
102,11,2006-07-12,2006,Fluminense,3,2,3,Juventude,1,M
2147,25,2011-09-22,2011,Vasco,1,1,1,Atletico-GO,1,E
2375,10,2012-07-18,2012,Sao Paulo,0,1,0,Vasco,1,V
4653,10,2018-06-06,2018,Palmeiras,2,0,3,Gremio,0,V
2642,37,2012-11-25,2012,Botafogo-RJ,2,3,0,Atletico-MG,1,V


## 2.4 Criando tabela de uma temporada específica a partir dos dados

In [330]:
# Filtrando os dados para o ano de 2024
df_2024 = df_tabela_total[df_tabela_total['ano'] == 2024]

# Agrupando por time e somando APENAS gols_pro, gols_contra e pontos e ordenando por pontos
df_ranking = df_2024.groupby('time')[['gols_pro', 'gols_contra', 'pontos']].sum().sort_values(by='pontos', ascending=False)

df_ranking

,gols_pro,gols_contra,pontos
time,,,
Botafogo-RJ,59,29,79
Palmeiras,60,33,73
Flamengo,61,42,70
Fortaleza,53,39,68
Internacional,53,36,65
Sao Paulo,53,43,59
Corinthians,54,45,56
Bahia,49,49,53
Cruzeiro,43,41,52


# 3. Ponto de comparação

In [328]:
df.value_counts('resultado', normalize=True) *100

resultado
M    49.085406
E    26.792999
V    24.121595
Name: proportion, dtype: float64